# Use Case 1: Load and Clean Chicago Crime Data

**Objective.** Build a data ingestion pipeline that reads the raw crime CSV, cleans it, adds new features and inserts the result into a database table through SQLAlchemy and PyMySQL.

**Files used** (all in `data/raw/`): `chicago_crime_dataset.csv` (the crimes) and `iucr_codes.csv` (the crime type for each IUCR code).

**Steps in this notebook**

| Task | What happens |
|---|---|
| 1 | Load the CSV into pandas, show the first 10 rows, the data types and the shape |
| 2 | Convert dates, handle missing values, standardise text columns |
| 3 | Create `Year`, `Month` and `DayOfWeek` |
| 4 | Use NumPy to get the missing-value percentage per column and decide what can be dropped |
| 5 | Design the `crime` table and write the cleaned data to the database |
| Questions | Unique crime types, date anomalies, outliers |

**Database.** If MySQL is configured in `.env` (see `.env.example`) the data goes to MySQL through PyMySQL. If MySQL is not running, the notebook uses a SQLite file with the same table design, so it still runs.

In [3]:
import io

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sqlalchemy import (BigInteger, Boolean, Column, DateTime, Float, Index, Integer, MetaData,
                        String, Table, inspect, text)
from sqlalchemy.schema import CreateTable

from db_config import RAW_DIR, get_engine, write_report

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Task 1: Load the dataset

`pd.read_csv` is used to load the file. The last two lines of the supplied CSV have an opening quote in the `location` column that is never closed, which would swallow the end of the file. The code below finds lines with an odd number of quotes and closes them in memory. The file on disk is not changed.

In [4]:
csv_path = RAW_DIR / "chicago_crime_dataset.csv"
lines = csv_path.read_text(encoding="utf-8-sig").splitlines()

broken_lines = [n for n, line in enumerate(lines, start=1) if line.count('"') % 2 == 1]
print("Lines with an unclosed quote:", broken_lines)

fixed = [line + '"' if line.count('"') % 2 == 1 else line for line in lines]
df_raw = pd.read_csv(io.StringIO("\n".join(fixed)))

Lines with an unclosed quote: [498, 499]


### 1a. First 10 rows

In [5]:
df_raw.head(10)

,id,case_number,date,block,iucr_code,location_desc,arrest,domestic,beat_num,district_code,ward_no,community_code,fbi_code,x_coordinate,y_coordinate,year,date_of_update,latitude,longitude,location
0,10508693,HZ250496,5/3/2016,013XX S SAWYER AVE,486,APARTMENT,True,True,1022,10,24,29,08B,1154907.0,1893681.0,2016,5/10/2016,41.864073,-87.706819,"(41.864073157, -87.706818608)"
1,10508695,HZ250409,5/3/2016,061XX S DREXEL AVE,486,RESIDENCE,False,True,313,3,20,42,08B,1183066.0,1864330.0,2016,5/10/2016,41.782922,-87.604363,"(41.782921527, -87.60436317)"
2,10508697,HZ250503,5/3/2016,053XX W CHICAGO AVE,470,STREET,False,False,1524,15,37,25,24,1140789.0,1904819.0,2016,5/10/2016,41.894908,-87.758372,"(41.894908283, -87.758371958)"
3,10508698,HZ250424,5/3/2016,049XX W FULTON ST,460,SIDEWALK,False,False,1532,15,28,25,08B,1143223.0,1901475.0,2016,5/10/2016,41.885687,-87.749516,"(41.885686845, -87.749515983)"
4,10508699,HZ250455,5/3/2016,003XX N LOTUS AVE,820,RESIDENCE,False,True,1523,15,28,25,6,1139890.0,1901675.0,2016,5/10/2016,41.886297,-87.761751,"(41.886297242, -87.761750709)"
5,10508702,HZ250447,5/3/2016,082XX S MARYLAND AVE,041A,STREET,False,False,631,6,8,44,04B,1183336.0,1850642.0,2016,5/10/2016,41.745354,-87.603799,"(41.745354023, -87.603798903)"
6,10508703,HZ250489,5/3/2016,027XX S STATE ST,460,CHA HALLWAY/STAIRWELL/ELEVATOR,False,False,133,1,3,35,08B,1176730.0,1886544.0,2016,5/10/2016,41.844024,-87.626923,"(41.844023772, -87.626923253)"
7,10508704,HZ250514,5/3/2016,002XX E 46TH ST,460,RESIDENCE PORCH/HALLWAY,False,False,215,2,3,38,08B,1178514.0,1874573.0,2016,5/10/2016,41.811134,-87.620741,"(41.811133958, -87.62074077)"
8,10508709,HZ250523,5/3/2016,014XX W DEVON AVE,460,SIDEWALK,False,False,2432,24,40,1,08B,1165696.0,1942616.0,2016,5/10/2016,41.998131,-87.665814,"(41.99813061, -87.665814038)"
9,10508982,HZ250667,5/3/2016,069XX S ASHLAND AVE,486,STREET,False,True,735,7,17,67,08B,1166876.0,1858796.0,2016,5/10/2016,41.768097,-87.663879,"(41.768096835, -87.663878589)"


### 1b. Schema and data types

In [6]:
df_raw.info()
print()
df_raw.dtypes.to_frame("dtype")

<class 'pandas.DataFrame'>
RangeIndex: 498 entries, 0 to 497
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              498 non-null    int64  
 1   case_number     498 non-null    str    
 2   date            498 non-null    str    
 3   block           498 non-null    str    
 4   iucr_code       498 non-null    str    
 5   location_desc   494 non-null    str    
 6   arrest          498 non-null    bool   
 7   domestic        498 non-null    bool   
 8   beat_num        498 non-null    int64  
 9   district_code   498 non-null    int64  
 10  ward_no         498 non-null    int64  
 11  community_code  498 non-null    int64  
 12  fbi_code        498 non-null    str    
 13  x_coordinate    469 non-null    float64
 14  y_coordinate    469 non-null    float64
 15  year            498 non-null    int64  
 16  date_of_update  498 non-null    str    
 17  latitude        469 non-null    float64
 18  l

,dtype
id,int64
case_number,str
date,str
block,str
iucr_code,str
location_desc,str
arrest,bool
domestic,bool
beat_num,int64
district_code,int64


### 1c. Number of rows and columns

In [7]:
n_rows, n_cols = df_raw.shape
print(f"Rows: {n_rows}")
print(f"Columns: {n_cols}")

Rows: 498
Columns: 20


## Task 2: Clean the dataset

Cleaning is done on a copy (`df`) so `df_raw` stays available for comparison.

### 2a. Convert the date columns to datetime

The dates look like `5/3/2016` (month/day/year). An explicit format is used, so a value in another format becomes `NaT` and is counted instead of being guessed. The checks on the result are used later to answer the date-anomaly question.

In [8]:
df = df_raw.copy()
df.columns = df.columns.str.strip().str.lower()

date_text = df["date"].astype("string").str.strip()
update_text = df["date_of_update"].astype("string").str.strip()

df["Date"] = pd.to_datetime(date_text, format="%m/%d/%Y", errors="coerce")
df["date_of_update"] = pd.to_datetime(update_text, format="%m/%d/%Y", errors="coerce")

print("Dates that could not be parsed:", int(df["Date"].isna().sum()))
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
df[["date", "Date", "date_of_update"]].head()

Dates that could not be parsed: 0
Date range: 2012-01-01 to 2016-05-04


,date,Date,date_of_update
0,5/3/2016,2016-05-03,2016-05-10
1,5/3/2016,2016-05-03,2016-05-10
2,5/3/2016,2016-05-03,2016-05-10
3,5/3/2016,2016-05-03,2016-05-10
4,5/3/2016,2016-05-03,2016-05-10


### 2b. Identify missing values

Both real missing values and empty strings are counted.

In [9]:
df = df.replace(r"^\s*$", pd.NA, regex=True)
missing = df.isna().sum()
missing_table = pd.DataFrame({"missing": missing, "missing_%": (missing / len(df) * 100).round(2)})
missing_table[missing_table["missing"] > 0]

,missing,missing_%
location_desc,4,0.80
x_coordinate,29,5.82
y_coordinate,29,5.82
latitude,29,5.82
longitude,29,5.82
location,29,5.82


### 2c. Handle missing values

* `location_desc` (where the crime happened) is filled with `UNKNOWN`. The row is still a valid crime.
* Coordinates and `location` are missing together in the same rows. They are **not** filled, because a made-up map position would be wrong. Counts by year, type or community do not need them.
* `location` is a text copy of `latitude` and `longitude`, so it is dropped. `year` repeats the year inside `date`, so it is checked against the date and then dropped. The new `Year` column replaces it.

In [10]:
year_matches = (df["year"] == df["Date"].dt.year).all()
print("Source 'year' column matches the year inside 'date':", year_matches)

df["location_desc"] = df["location_desc"].fillna("UNKNOWN")
df = df.drop(columns=["location", "year", "date"])
print("Missing values left per column (non-zero only):")
print(df.isna().sum()[lambda s: s > 0])

Source 'year' column matches the year inside 'date': True
Missing values left per column (non-zero only):
x_coordinate    29
y_coordinate    29
latitude        29
longitude       29
dtype: int64


### 2d. Standardise the categorical columns

* Strip leading and trailing spaces and collapse repeated spaces.
* Use **upper case** everywhere, so `Street` and `STREET` become one value.
* The IUCR reference file has a header called `PRIMARY_TYPE ` with a trailing space. Column names are stripped for that reason.
* The crime type and description are added from the IUCR file by joining on the IUCR code.

In [11]:
text_cols = ["case_number", "block", "iucr_code", "location_desc", "fbi_code"]
before = df["location_desc"].nunique()

for col in text_cols:
    df[col] = (df[col].astype("string").str.strip()
               .str.replace(r"\s+", " ", regex=True).str.upper())

iucr = pd.read_csv(RAW_DIR / "iucr_codes.csv", dtype=str)
iucr.columns = iucr.columns.str.strip().str.lower().str.replace(" ", "_")
for col in ["iucr", "primary_type", "description"]:
    iucr[col] = iucr[col].str.strip().str.replace(r"\s+", " ", regex=True).str.upper()
iucr = iucr.drop_duplicates("iucr")

df = (df.merge(iucr[["iucr", "primary_type", "description"]], left_on="iucr_code", right_on="iucr", how="left")
        .drop(columns="iucr"))

print(f"Distinct location_desc values: {before} before, {df['location_desc'].nunique()} after")
print("Crimes whose IUCR code is not in the reference file:", int(df["primary_type"].isna().sum()))
df[["case_number", "block", "iucr_code", "primary_type", "description", "location_desc", "fbi_code"]].head()

Distinct location_desc values: 48 before, 48 after
Crimes whose IUCR code is not in the reference file: 0


,case_number,block,iucr_code,primary_type,description,location_desc,fbi_code
0,HZ250496,013XX S SAWYER AVE,486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,08B
1,HZ250409,061XX S DREXEL AVE,486,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE,08B
2,HZ250503,053XX W CHICAGO AVE,470,PUBLIC PEACE VIOLATION,RECKLESS CONDUCT,STREET,24
3,HZ250424,049XX W FULTON ST,460,BATTERY,SIMPLE,SIDEWALK,08B
4,HZ250455,003XX N LOTUS AVE,820,THEFT,$500 AND UNDER,RESIDENCE,6


## Task 3: Generate new features

`Year`, `Month` and `DayOfWeek` are taken from the `Date` column.

In [12]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["DayOfWeek"] = df["Date"].dt.day_name()
df[["Date", "Year", "Month", "DayOfWeek"]].head(10)

,Date,Year,Month,DayOfWeek
0,2016-05-03,2016,5,Tuesday
1,2016-05-03,2016,5,Tuesday
2,2016-05-03,2016,5,Tuesday
3,2016-05-03,2016,5,Tuesday
4,2016-05-03,2016,5,Tuesday
5,2016-05-03,2016,5,Tuesday
6,2016-05-03,2016,5,Tuesday
7,2016-05-03,2016,5,Tuesday
8,2016-05-03,2016,5,Tuesday
9,2016-05-03,2016,5,Tuesday


## Task 4: Use NumPy

### 4a. Percentage of missing values per column

The percentage is calculated on the **raw** data, so it describes the file as supplied. NumPy turns the missing-value flags into an array and takes the mean of each column.

In [13]:
missing_pct = np.round(df_raw.isna().to_numpy().mean(axis=0) * 100, 2)
missing_pct_table = pd.Series(missing_pct, index=df_raw.columns, name="missing_%")
missing_pct_table.to_frame()

,missing_%
id,0.00
case_number,0.00
date,0.00
block,0.00
iucr_code,0.00
location_desc,0.80
arrest,0.00
domestic,0.00
beat_num,0.00
district_code,0.00


### 4b. Can columns with more than 50% missing values be dropped?

In [14]:
over_50 = missing_pct_table.index[np.where(missing_pct > 50)[0]].tolist()
highest = missing_pct_table.idxmax()
print("Columns with more than 50% missing:", over_50 if over_50 else "none")
print(f"Highest missing percentage: {missing_pct_table.max()}% ({highest})")
if not over_50:
    print("Decision: no column crosses the 50% threshold, so no column has to be dropped for that reason.")

Columns with more than 50% missing: none
Highest missing percentage: 5.82% (x_coordinate)
Decision: no column crosses the 50% threshold, so no column has to be dropped for that reason.


## Task 5: Insert the cleaned data into the database

### 5a. Design the `crime` table

One row per crime. The key points of the design:

| Column group | Design choice |
|---|---|
| `id` | Primary key, taken from the source data |
| `crime_date`, `crime_year`, `crime_month`, `crime_day_name` | Date and the three features from Task 3. Names are lower case with a prefix, to avoid MySQL keywords such as `YEAR` |
| `iucr_code`, `primary_type`, `description`, `fbi_code` | What kind of crime it was |
| `arrest`, `domestic` | Boolean flags |
| `beat_num`, `district_code`, `ward_no`, `community_code` | Where it happened, as numeric codes |
| coordinates | Nullable, because about 6% are missing |
| Indexes | On year, crime type and community, the columns used for grouping later |

In [15]:
metadata = MetaData()
crime = Table(
    "crime", metadata,
    Column("id", BigInteger, primary_key=True, autoincrement=False),
    Column("case_number", String(16), nullable=False),
    Column("crime_date", DateTime, nullable=False),
    Column("crime_year", Integer, nullable=False),
    Column("crime_month", Integer, nullable=False),
    Column("crime_day_name", String(12), nullable=False),
    Column("block", String(64), nullable=False),
    Column("iucr_code", String(8), nullable=False),
    Column("primary_type", String(64)),
    Column("description", String(128)),
    Column("location_desc", String(64)),
    Column("fbi_code", String(4)),
    Column("arrest", Boolean, nullable=False),
    Column("domestic", Boolean, nullable=False),
    Column("beat_num", Integer),
    Column("district_code", Integer),
    Column("ward_no", Integer),
    Column("community_code", Integer),
    Column("x_coordinate", Float),
    Column("y_coordinate", Float),
    Column("latitude", Float),
    Column("longitude", Float),
    Column("date_of_update", DateTime),
    Index("ix_crime_year", "crime_year"),
    Index("ix_crime_primary_type", "primary_type"),
    Index("ix_crime_community", "community_code"),
)

engine, backend = get_engine()
print(CreateTable(crime).compile(engine))

Using SQLite database chicago_crime_analysis.db (MySQL is not configured or not running).

CREATE TABLE crime (
	id BIGINT NOT NULL, 
	case_number VARCHAR(16) NOT NULL, 
	crime_date DATETIME NOT NULL, 
	crime_year INTEGER NOT NULL, 
	crime_month INTEGER NOT NULL, 
	crime_day_name VARCHAR(12) NOT NULL, 
	block VARCHAR(64) NOT NULL, 
	iucr_code VARCHAR(8) NOT NULL, 
	primary_type VARCHAR(64), 
	description VARCHAR(128), 
	location_desc VARCHAR(64), 
	fbi_code VARCHAR(4), 
	arrest BOOLEAN NOT NULL, 
	domestic BOOLEAN NOT NULL, 
	beat_num INTEGER, 
	district_code INTEGER, 
	ward_no INTEGER, 
	community_code INTEGER, 
	x_coordinate FLOAT, 
	y_coordinate FLOAT, 
	latitude FLOAT, 
	longitude FLOAT, 
	date_of_update DATETIME, 
	PRIMARY KEY (id)
)




### 5b. Write the data with SQLAlchemy and PyMySQL

SQLAlchemy creates the table. `DataFrame.to_sql` then inserts the rows. With MySQL, SQLAlchemy uses the PyMySQL driver (`mysql+pymysql://...`). The table is dropped and rebuilt first, so running the notebook twice gives the same result.

In [16]:
df_db = df.rename(columns={"Date": "crime_date", "Year": "crime_year",
                           "Month": "crime_month", "DayOfWeek": "crime_day_name"})
df_db = df_db[[c.name for c in crime.columns]]

assert df_db["id"].is_unique, "id must be unique because it is the primary key"

metadata.drop_all(engine)
metadata.create_all(engine)
df_db.to_sql("crime", engine, if_exists="append", index=False, chunksize=200)

with engine.connect() as conn:
    rows_in_db = conn.execute(text("SELECT COUNT(*) FROM crime")).scalar()
print(f"Backend: {backend}")
print(f"Rows in DataFrame: {len(df_db)}  |  Rows in database table 'crime': {rows_in_db}")

Backend: sqlite
Rows in DataFrame: 498  |  Rows in database table 'crime': 498


### 5c. Check the table by reading it back

In [17]:
check = pd.read_sql("SELECT id, case_number, crime_date, crime_year, crime_month, crime_day_name, primary_type, arrest FROM crime ORDER BY id LIMIT 5", engine)
display(check)
pd.DataFrame(inspect(engine).get_columns("crime"))[["name", "type", "nullable"]]

,id,case_number,crime_date,crime_year,crime_month,crime_day_name,primary_type,arrest
0,20859,HW254987,2013-05-01 00:00:00.000000,2013,5,Wednesday,HOMICIDE,1
1,22451,HZ250365,2016-05-03 00:00:00.000000,2016,5,Tuesday,HOMICIDE,0
2,22452,HZ249724,2016-05-03 00:00:00.000000,2016,5,Tuesday,HOMICIDE,0
3,10077106,HY266148,2015-05-19 00:00:00.000000,2015,5,Tuesday,ASSAULT,1
4,10301916,HY469211,2015-01-01 00:00:00.000000,2015,1,Thursday,CRIMINAL SEXUAL ASSAULT,1


,name,type,nullable
0,id,BIGINT,False
1,case_number,VARCHAR(16),False
2,crime_date,DATETIME,False
3,crime_year,INTEGER,False
4,crime_month,INTEGER,False
5,crime_day_name,VARCHAR(12),False
6,block,VARCHAR(64),False
7,iucr_code,VARCHAR(8),False
8,primary_type,VARCHAR(64),True
9,description,VARCHAR(128),True


## Questions

### Q1. How many unique crime types exist in the dataset?

In [18]:
n_types = df["primary_type"].nunique()
type_counts = df["primary_type"].value_counts()
n_iucr_used = df["iucr_code"].nunique()
ref_types = iucr["primary_type"].nunique()
print(f"Unique crime types (primary_type): {n_types}")
print(f"Unique IUCR codes used: {n_iucr_used}")
print(f"For comparison, the IUCR reference file defines {ref_types} crime types.")
type_counts.to_frame("crimes")

Unique crime types (primary_type): 23
Unique IUCR codes used: 97
For comparison, the IUCR reference file defines 33 crime types.


,crimes
primary_type,
THEFT,105
BATTERY,85
CRIMINAL DAMAGE,49
OTHER OFFENSE,39
DECEPTIVE PRACTICE,38
ASSAULT,35
NARCOTICS,32
ROBBERY,30
CRIMINAL TRESPASS,17


### Q2. Are there anomalies in the date formats?

In [19]:
date_text_raw = df_raw["date"].astype("string").str.strip()
update_text_raw = df_raw["date_of_update"].astype("string").str.strip()
pattern = r"^\d{1,2}/\d{1,2}/\d{4}$"

checks = {
    "dates not in M/D/YYYY format": int((~date_text_raw.str.match(pattern)).sum()),
    "dates that could not be parsed": int(df["Date"].isna().sum()),
    "dates that contain a time of day": int(date_text_raw.str.contains(":").sum()),
    "dates in the future": int((df["Date"] > pd.Timestamp.today()).sum()),
    "update date earlier than crime date": int((df["date_of_update"] < df["Date"]).sum()),
    "source 'year' different from the year in 'date'": int((df_raw["year"] != df["Date"].dt.year).sum()),
}
no_zero_pad = int((date_text_raw.str.match(pattern) & ~date_text_raw.str.match(r"^\d{2}/\d{2}/\d{4}$")).sum())
checks["dates without zero padding (5/3/2016 instead of 05/03/2016)"] = no_zero_pad
pd.Series(checks, name="count").to_frame()

,count
dates not in M/D/YYYY format,0
dates that could not be parsed,0
dates that contain a time of day,0
dates in the future,0
update date earlier than crime date,0
source 'year' different from the year in 'date',0
dates without zero padding (5/3/2016 instead of 05/03/2016),497


### Q3. Are there obvious outliers?

The IQR rule is used: a value is an outlier if it is below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR`. The checks cover the map coordinates, the days between the crime and its last update, and the number of crimes per day.

In [20]:
def iqr_outliers(series):
    series = series.dropna()
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((series < low) | (series > high)).sum()), low, high

candidates = {
    "x_coordinate": df["x_coordinate"],
    "y_coordinate": df["y_coordinate"],
    "latitude": df["latitude"],
    "longitude": df["longitude"],
    "days from crime to update": (df["date_of_update"] - df["Date"]).dt.days,
    "crimes per day": df.groupby("Date").size(),
}
rows = []
for name, series in candidates.items():
    count, low, high = iqr_outliers(series)
    rows.append({"measure": name, "values checked": int(series.notna().sum()), "outliers (IQR)": count,
                 "lower fence": round(low, 2), "upper fence": round(high, 2),
                 "max": round(series.max(), 2)})
outlier_table = pd.DataFrame(rows)

in_chicago = df["latitude"].between(41.6, 42.1) & df["longitude"].between(-87.95, -87.5)
outside = int((df["latitude"].notna() & ~in_chicago).sum())
print("Coordinates outside the Chicago bounding box:", outside)
outlier_table

Coordinates outside the Chicago bounding box: 0


,measure,values checked,outliers (IQR),lower fence,upper fence,max
0,x_coordinate,469,0,1116781.50,1212833.50,1202048.00
1,y_coordinate,469,0,1791096.50,1975924.50,1950346.00
2,latitude,469,0,41.58,42.09,42.02
3,longitude,469,0,-87.85,-87.49,-87.54
4,days from crime to update,498,59,7.00,7.00,1592.00
5,crimes per day,46,11,1.00,1.00,298.00


## Answers

The text below is generated from the results above and is also saved to `data/reports/use_case_1/use_case_1_answers.md`.

In [21]:
top_type, top_n = type_counts.index[0], int(type_counts.iloc[0])
days = candidates["crimes per day"]
busiest = days.sort_values(ascending=False).head(2)
busy_share = busiest.sum() / len(df) * 100
out = outlier_table.set_index("measure")["outliers (IQR)"]

parsed_ok = len(df_raw) - checks["dates that could not be parsed"]
q2_text = (
    f"Only minor ones. The dates are text such as 5/3/2016 (month/day/year) and {parsed_ok} of {len(df_raw)} parse with the "
    "explicit format %m/%d/%Y. Check results: " + "; ".join(f"{k}: {v}" for k, v in checks.items()) + ". "
    f"The only non-standard pattern is the missing zero padding ({no_zero_pad} of {len(df_raw)} values), which does not affect parsing. "
    f"Because no date has a time of day ({checks['dates that contain a time of day']} do), every Hour is 0, which matters in Use Case 3. "
    f"A separate file problem: the last lines of the CSV (file lines {broken_lines}) have an unclosed quote in the location column."
)
q3_text = (
    f"Yes, in the following measures. Outliers by the IQR rule: "
    + "; ".join(f"{k}: {int(v)}" for k, v in out.items()) + ". "
    f"{outside} coordinates fall outside the Chicago bounding box (latitude 41.6 to 42.1, longitude -87.95 to -87.5). "
    f"The two busiest dates ({', '.join(str(d.date()) for d in busiest.index)}) hold {busy_share:.2f}% of all crimes, "
    f"which shows the file is a sample that covers a few dates heavily, not the whole city for a full period."
)
sections = [
    ("Task 1: Load", f"The file has {n_rows} rows and {n_cols} columns. See the notebook for the first 10 rows and the data types."),
    ("Task 2: Clean", f"Dates were converted with %m/%d/%Y. location_desc missing values were set to UNKNOWN. Coordinates were left empty where missing. Text columns were stripped and set to upper case."),
    ("Task 4: Missing values", f"Highest missing percentage is {missing_pct_table.max()}% ({highest}). Columns above 50% missing: {', '.join(over_50) if over_50 else 'none'}, so none must be dropped."),
    ("Task 5: Database", f"{rows_in_db} rows were written to the crime table ({backend})."),
    ("Question 1: How many unique crime types exist?", f"{n_types} unique crime types (primary_type) appear in the {len(df)} crime records. {top_type} is the largest with {top_n} records. The IUCR reference file defines {ref_types} types, and {n_iucr_used} distinct IUCR codes are used."),
    ("Question 2: Are there anomalies in date formats?", q2_text),
    ("Question 3: Are there obvious outliers?", q3_text),
]
write_report("use_case_1", "use_case_1_answers.md", "Use Case 1 answers", sections)
for heading, body in sections[4:]:
    display(Markdown(f"**{heading}**\n\n{body}"))

**Question 1: How many unique crime types exist?**

23 unique crime types (primary_type) appear in the 498 crime records. THEFT is the largest with 105 records. The IUCR reference file defines 33 types, and 97 distinct IUCR codes are used.

**Question 2: Are there anomalies in date formats?**

Only minor ones. The dates are text such as 5/3/2016 (month/day/year) and 498 of 498 parse with the explicit format %m/%d/%Y. Check results: dates not in M/D/YYYY format: 0; dates that could not be parsed: 0; dates that contain a time of day: 0; dates in the future: 0; update date earlier than crime date: 0; source 'year' different from the year in 'date': 0; dates without zero padding (5/3/2016 instead of 05/03/2016): 497. The only non-standard pattern is the missing zero padding (497 of 498 values), which does not affect parsing. Because no date has a time of day (0 do), every Hour is 0, which matters in Use Case 3. A separate file problem: the last lines of the CSV (file lines [498, 499]) have an unclosed quote in the location column.

**Question 3: Are there obvious outliers?**

Yes, in the following measures. Outliers by the IQR rule: x_coordinate: 0; y_coordinate: 0; latitude: 0; longitude: 0; days from crime to update: 59; crimes per day: 11. 0 coordinates fall outside the Chicago bounding box (latitude 41.6 to 42.1, longitude -87.95 to -87.5). The two busiest dates (2016-05-04, 2016-05-03) hold 88.96% of all crimes, which shows the file is a sample that covers a few dates heavily, not the whole city for a full period.